# TOpic covered
TWO APPROACHES: Deterministic approach vs Model-based

1.Build-in:PII detetction middleware
2.Build-in:HITL middleware
3.Custom:BAG and AAG (Before and After Agent Guardrails{input filtering & output safety})
4.Layered/Combined Guardrails
5.Real world use case:Helathcare Chatbot

In [1]:
from dotenv import load_dotenv
load_dotenv()
import os
from getpass import getpass

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [4]:
#two approaches
#Deterministic Approach
def deterministic_guardrails(text:str)->bool:
    """Returns True if content is blocked"""
    banned_keywords=["hack","exploit","malware","bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

text_input=[
    "How do i hack a server",
    "What is the capital of France",
    "Explain about how malware spreads and hwo to execute",
]

for input in text_input:
    blocked=deterministic_guardrails(input)
    status="Blocked" if blocked else "Allowed"
    print(f"{status} : {input}")

Blocked : How do i hack a server
Allowed : What is the capital of France
Blocked : Explain about how malware spreads and hwo to execute


In [9]:
#Model based approach
from langchain_groq import ChatGroq
def model_guardrails(text:str)->str:
    """Use an LLM to evaluate if the text is SAFE or UNSAFE"""
    model=ChatGroq(model="qwen/qwen3.6-27b")
    prompt = f"""You are a strict content safety filter. Reply UNSAFE if the input requests any of:
- Techniques to create, execute, deploy, or spread malware/viruses
- Steps to exploit, hack, or gain unauthorized access to systems or servers
- Any operational/actionable detail that could enable a cyberattack, even if framed as "explaining" or "educational"

Ambiguous or partially-technical requests about attacks should default to UNSAFE unless clearly framed for defense (e.g., detection, prevention).

Reply with only SAFE or UNSAFE, nothing else.
Input: {text}"""
    result=model.invoke([{"role":"user","content":prompt}],reasoning_effort="none")
    return result.content.strip()

for input in text_input:
    blocked=model_guardrails(input)
    status="UNSAFE" if "UNSAFE" in blocked.upper() else "SAFE"
    print(f"{status} : {input}")


UNSAFE : How do i hack a server
SAFE : What is the capital of France
UNSAFE : Explain about how malware spreads and hwo to execute


In [20]:
model = ChatGroq(
    model="qwen/qwen3.6-27b",
    reasoning_effort="none",
)

In [25]:
#Build-in guardrail=PII Detection middleware
#stratagies:
# redact->[REDACTED_EMAIL]
# mask->****-****-****-1234
# hash->a8f512kt....
# block->raises an exception

from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware,HumanInTheLoopMiddleware
from langchain_core.tools import tool
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver

@tool
def customer_lookup(query:str)->str:
    """Look up customer information"""
    return f"Customer record found for query: {query}"

model = ChatGroq(
    model="qwen/qwen3.6-27b",
    reasoning_effort="none",
)

agent = create_agent(
    model=model,
    tools=[customer_lookup],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),

    ]
)
print("Agent with PII Middleware created successfully")

Agent with PII Middleware created successfully


In [22]:
#test pii middeware
result=agent.invoke({
    "messages":[{
        "role":"user",
        "content":"My email is ahmed@gmail.com and my credit ccard is 1234-1234-234-1234. can you help me"
    }]
})

print(result["messages"][-1].content)

I cannot assist with requests involving sensitive personal information such as credit card numbers or private email addresses. I am designed to prioritize user safety and privacy, so I do not process, store, or verify financial details or personal contact information.

If you are looking for general information about account management, security best practices, or how to protect your personal data, I would be happy to provide helpful and safe guidance on those topics.


In [23]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my credit ccard is 1234-1234-234-1234. can you help me', additional_kwargs={}, response_metadata={}, id='e33e0e6a-bd87-443d-b72c-497a4cf12198'),
  AIMessage(content='I cannot assist with requests involving sensitive personal information such as credit card numbers or private email addresses. I am designed to prioritize user safety and privacy, so I do not process, store, or verify financial details or personal contact information.\n\nIf you are looking for general information about account management, security best practices, or how to protect your personal data, I would be happy to provide helpful and safe guidance on those topics.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 308, 'total_tokens': 393, 'completion_time': 0.161801918, 'completion_tokens_details': None, 'prompt_time': 0.0225512, 'prompt_tokens_details': None, 'queue_time': 0.052298942, 'total_time'

In [24]:
#test api kety
try:
    result=agent.invoke({
    "messages":[{
        "role":"user",
        "content":"My key sk-qwertyuiiopasdfghjklzxcvbnm-1234567890"
        }]
    })
except Exception as e:
    print(f"Blocked as expected: {e}")

In [34]:
#HITL
model = ChatGroq(
    model="qwen/qwen3.6-27b",
    reasoning_effort="none",
)

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

# Create agent with HITL middleware
hitl_agent = create_agent(
    model=model,
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,   # Require approval
                "search_web": False,      # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

print("Human-in-the-Loop agent created!")

Human-in-the-Loop agent created!


In [35]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)

=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='e150e683-3537-4d80-b764-80236adfce90'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '36zmgwyp0', 'function': {'arguments': '{"body":"Hi Team,\\n\\nPlease find the Q4 results attached. Let me know if you have any questions.\\n\\nBest regards,\\nYour Name","subject":"Q4 Results Overview","to":"team@company.com"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 431, 'total_tokens': 515, 'completion_time': 0.15972059, 'completion_tokens_details': None, 'prompt_time': 0.030500044, 'prompt_tokens_details': None, 'queue_time': 0.047165058, 'total_time': 0.190220634}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_49d6b1859d', 'service_tier': 'on_demand', 'reasoning_effort': 'none', 'finish_reason': 'to

In [36]:
# Step 2: Human reviews and APPROVES
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
The email with the subject "Q4 Results Overview" has been successfully sent to team@company.com.


In [37]:
# Step 3: Alternative — Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

=== Rejected! Final response ===
I'm sorry, but I cannot delete records from the database. Please ensure that you have administrative privileges and proceed with caution when performing such operations. If you need assistance with anything else, feel free to ask!


**Best for: BAH**
- Keyword/content filtering
- Authentication checks
- Rate limiting
- Blocking specific categories of requests

**Best for: AGH**
- Model-based safety evaluation of outputs
- Compliance scanning (e.g. legal, medical, financial disclaimers)
- Quality validation
- Removing sensitive info that slipped through

In [39]:
#Before agent hook
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"🚫 Blocked — keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None


@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


# Create agent with content filter
filtered_agent = create_agent(
    model=model,
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

print("Content filter agent created!")

Content filter agent created!


In [41]:
# Test 1: Safe request — should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("✅ Safe request response:")
print(result["messages"][-1].content)

✅ Safe request response:
**Machine Learning (ML)** is a subset of artificial intelligence (AI) that provides systems the ability to automatically learn and improve from experience without being explicitly programmed.

Here is a breakdown of what that means:

### 1. The Core Concept
In traditional programming, a human writes specific rules and logic (e.g., "If X happens, then do Y"). The computer follows these instructions exactly.

In machine learning, the programmer provides the computer with **data** and a **goal**. The computer then analyzes the data to find patterns and creates its own rules to achieve that goal.

### 2. How It Works
The process generally involves three steps:
1.  **Training:** The algorithm is fed a large dataset (examples).
2.  **Learning:** The algorithm identifies patterns, correlations, and structures within that data.
3.  **Prediction/Decision:** Once trained, the model can make predictions or decisions when presented with new, unseen data.

### 3. Main Types

In [40]:
# Test 2: Unsafe request — should be blocked
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print("🚫 Unsafe request response:")
print(result["messages"][-1].content)

🚫 Blocked — keyword detected: 'hack'
🚫 Unsafe request response:
I cannot process requests containing inappropriate content. Please rephrase your request.


#Combine/Layered middlewre

In [44]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain.agents import create_agent
from langchain_core.tools import tool

class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = model

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            print("⚠️  Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None


@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"


safe_agent = create_agent(
    model=model,
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("Output safety agent created!")

Output safety agent created!


In [47]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def search_tool(query:str)->str:
    """Search for information"""
    return f"Search for {query}"
@tool
def send_email_tool(to:str,body:str)->str:
    """Send email"""
    return f"Send email {to}"

production_agent=create_agent(
    model=model,
    tools=[search_tool,send_email],
    middleware=[
        #1
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware"]),
        #2
        PIIMiddleware("credit_card",strategy="mask",apply_to_input=True),
        #3
        HumanInTheLoopMiddleware(
                    interrupt_on={"send_email_tool": True, "search_tool": False}
                ),
        #4
        PIIMiddleware("email",strategy="redact",apply_to_output=True),
        #5
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver()
)
print("🏭 Production-grade agent with 5-layer guardrails created!")

🏭 Production-grade agent with 5-layer guardrails created!
